In [6]:
# hasse_sl5_adjusted_figs.py
# Generates sl5 body-order Hasse diagrams (e=9,10,11) with:
# - partitions: no box, larger bold text
# - shorter arrows (extra short if both ends are partitions)
# Outputs PNG + tight-page PDF to ./hasse_sl5_adjusted/

from PIL import Image, ImageDraw, ImageFont, ImageOps
import itertools, numpy as np, collections, os, math

# ---------- I/O ----------
OUTPUT_DIR = "./hasse_sl5_adjusted"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------- core combinatorics ----------
def compositions_of_n(n):
    res=[]
    def rec(prefix, rem):
        if rem==0:
            res.append(tuple(prefix)); return
        for a in range(1, rem+1):
            rec(prefix+[a], rem-a)
    rec([], n)
    return res

def all_weights_for_mu(mu, r, e):
    pts=set()
    j=len(mu)
    for runners in itertools.combinations(range(1,e+1), j):
        beta=[]
        for runner_pos, mult in zip(runners, mu):
            for k in range(mult):
                beta.append(runner_pos + k*e)
        beta.sort(reverse=True)
        lam=[beta[i] - (r - (i+1)) for i in range(r)]
        w = tuple(lam[i]-lam[i+1] for i in range(r-1))
        pts.add(w)
    return np.array(sorted(list(pts)), dtype=float)

def max_vector(mu, r, e):
    arr = all_weights_for_mu(mu, r, e)
    return arr.max(axis=0)

def ge_maxvec(a,b, tol=1e-9):
    return np.all(a >= b - tol)

def is_partition_comp(mu):
    return all(mu[i] >= mu[i+1] for i in range(len(mu)-1))

def build_max_order(r,e):
    comps = compositions_of_n(r)
    max_map = {mu: max_vector(mu,r,e) for mu in comps}
    covers=[]
    for mu in comps:
        for nu in comps:
            if mu==nu: continue
            if not ge_maxvec(max_map[mu], max_map[nu]): 
                continue
            minimal=True
            for tau in comps:
                if tau==mu or tau==nu: continue
                if ge_maxvec(max_map[mu], max_map[tau]) and ge_maxvec(max_map[tau], max_map[nu]):
                    minimal=False; break
            if minimal:
                covers.append((mu,nu))
    groups=collections.defaultdict(list)
    for mu,v in max_map.items():
        key=tuple(v.tolist())
        groups[key].append(mu)
    comp_to_group={mu: key for key, mus in groups.items() for mu in mus}
    gedges=set()
    for mu,nu in covers:
        gmu=comp_to_group[mu]; gnu=comp_to_group[nu]
        if gmu!=gnu:
            gedges.add((gmu,gnu))
    def key_group(mv):
        arr=np.array(mv)
        return (-arr.sum(), -arr.max(), mv)
    groups_sorted=sorted(groups.keys(), key=key_group)
    group_members = { mv: sorted(groups[mv], key=lambda x:(len(x),x)) for mv in groups_sorted }
    gedges_sorted=sorted(gedges, key=lambda p:(key_group(p[0]), key_group(p[1])))
    return {"r":r,"e":e,"groups_sorted":groups_sorted,"group_members":group_members,"group_edges":gedges_sorted}

# ---------- layout ----------
def build_levels(nodes, edges):
    incoming={n:set() for n in nodes}
    outgoing={n:set() for n in nodes}
    for u,v in edges:
        outgoing[u].add(v); incoming[v].add(u)
    sources=[n for n in nodes if len(incoming[n])==0]
    level={}
    from collections import deque
    dq=deque(sources)
    for s in sources: level[s]=0
    while dq:
        u=dq.popleft()
        for v in outgoing[u]:
            cand=level[u]+1
            if v not in level or cand>level[v]:
                level[v]=cand; dq.append(v)
    level_to_nodes=collections.defaultdict(list)
    for n,l in level.items():
        level_to_nodes[l].append(n)
    for l in level_to_nodes:
        level_to_nodes[l].sort(key=lambda mv:(-sum(mv), mv))
    return level, level_to_nodes

def format_node_label(members, per_line=3):
    def fmt(mu):
        s=str(mu)
        return "*"+s if is_partition_comp(mu) else s
    tokens=[fmt(mu) for mu in members]
    lines=[]
    for i in range(0,len(tokens),per_line):
        lines.append(", ".join(tokens[i:i+per_line]))
    return lines

def layout_for_rank(r,e, x_spacing=210, y_spacing=115, margin_x=110, margin_y=55,
                    line_height=13, char_w=7, pad_x=6, pad_y=6):
    summary = build_max_order(r,e)
    nodes = summary["groups_sorted"]
    edges = summary["group_edges"]
    members = summary["group_members"]
    level, level_to_nodes = build_levels(nodes, edges)
    pos_center={}
    for lev, node_list in level_to_nodes.items():
        nL = len(node_list)
        xs=[0] if nL==1 else list(np.linspace(-(nL-1)/2.0, (nL-1)/2.0, nL))
        for xoff, mv in zip(xs, node_list):
            cx = margin_x + x_spacing * xoff + x_spacing*3
            cy = margin_y + y_spacing * lev
            pos_center[mv]=(cx,cy)
    node_boxes={}; all_rects=[]
    for mv in nodes:
        lines = format_node_label(members[mv], per_line=3)
        max_chars = max((len(L) for L in lines), default=1)
        rect_w = max(120, pad_x*2 + max_chars*char_w)
        rect_h = pad_y*2 + line_height*len(lines)
        cx,cy = pos_center[mv]
        x = cx - rect_w/2; y = cy - rect_h/2
        parts_only = all(is_partition_comp(mu) for mu in members[mv])
        node_boxes[mv]={"cx":cx,"cy":cy,"x":x,"y":y,"w":rect_w,"h":rect_h,
                        "lines":lines,"line_height":line_height,"parts_only":parts_only}
        all_rects.append((x,y,rect_w,rect_h))
    min_x = min(x for (x,y,w,h) in all_rects)-50
    min_y = min(y for (x,y,w,h) in all_rects)-50
    max_x = max(x+w for (x,y,w,h) in all_rects)+50
    max_y = max(y+h for (x,y,w,h) in all_rects)+50
    width  = max_x - min_x; height = max_y - min_y
    for mv in node_boxes:
        b=node_boxes[mv]
        b["x"] -= min_x; b["y"] -= min_y
        b["cx"]-= min_x; b["cy"]-= min_y
    edges_xy=[]
    for (u,v) in edges:
        ub=node_boxes[u]; vb=node_boxes[v]
        x1=ub["cx"]; y1=ub["y"]+ub["h"]
        x2=vb["cx"]; y2=vb["y"]
        edges_xy.append((u,v,x1,y1,x2,y2))
    return node_boxes, edges_xy, width, height

# ---------- drawing ----------
def shrink_segment(x1,y1,x2,y2, shrink_start, shrink_end):
    dx,dy = x2-x1, y2-y1
    L = math.hypot(dx,dy)
    if L <= (shrink_start + shrink_end + 1):
        return x1,y1,x2,y2
    ux,uy = dx/L, dy/L
    x1n = x1 + ux*shrink_start
    y1n = y1 + uy*shrink_start
    x2n = x2 - ux*shrink_end
    y2n = y2 - uy*shrink_end
    return x1n,y1n,x2n,y2n

def draw_arrow(draw, x1,y1,x2,y2, scale=1.5, color=(0,0,0), width=2):
    draw.line((x1,y1,x2,y2), fill=color, width=width)
    dx, dy = x2-x1, y2-y1
    L = math.hypot(dx,dy) or 1.0
    ux, uy = dx/L, dy/L
    ah_len = 7*scale
    ah_wid = 4.5*scale
    bx = x2 - ux*ah_len
    by = y2 - uy*ah_len
    px, py = -uy, ux
    p1 = (bx + px*ah_wid, by + py*ah_wid)
    p2 = (bx - px*ah_wid, by - py*ah_wid)
    draw.polygon([ (x2,y2), p1, p2 ], fill=color)

def render_adjusted_sl5(e):
    r=5
    node_boxes, edges_xy, width, height = layout_for_rank(
        r,e, x_spacing=210, y_spacing=115, margin_x=110, margin_y=55,
        line_height=13, char_w=7, pad_x=6, pad_y=6
    )
    scale = 1.6
    W=int(width*scale); H=int(height*scale)
    img = Image.new("RGB", (W,H), (255,255,255))
    draw = ImageDraw.Draw(img)

    # fonts (try common Mac/Linux fonts; fall back to default)
    normal_font_path = None
    bold_font_path   = None
    for fp in [
        "/System/Library/Fonts/Menlo.ttc",
        "/Library/Fonts/Menlo.ttc",
        "/System/Library/Fonts/Supplemental/Courier New.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
    ]:
        if os.path.exists(fp):
            normal_font_path = fp; break
    for fp in [
        "/System/Library/Fonts/Supplemental/Courier New Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSansMono-Bold.ttf",
        "/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf",
        "/usr/share/fonts/truetype/liberation/LiberationMono-Bold.ttf",
    ]:
        if os.path.exists(fp):
            bold_font_path = fp; break
    try:
        font_norm = ImageFont.truetype(normal_font_path, 17) if normal_font_path else ImageFont.load_default()
    except:
        font_norm = ImageFont.load_default()
    try:
        font_part = ImageFont.truetype(bold_font_path or normal_font_path, 21) if (bold_font_path or normal_font_path) else ImageFont.load_default()
    except:
        font_part = ImageFont.load_default()

    # edges (shorter; extra shrink if both ends are partitions)
    for (u,v,x1,y1,x2,y2) in edges_xy:
        parts_u = node_boxes[u]["parts_only"]
        parts_v = node_boxes[v]["parts_only"]
        shrink_start = 24
        shrink_end   = 24
        if parts_u and parts_v:
            shrink_start += 16
            shrink_end   += 16
        X1,Y1,X2,Y2 = shrink_segment(x1*scale,y1*scale,x2*scale,y2*scale,shrink_start,shrink_end)
        draw_arrow(draw, X1,Y1,X2,Y2, scale=scale, color=(0,0,0), width=2)

    # nodes and labels
    for mv,b in node_boxes.items():
        x=b["x"]*scale; y=b["y"]*scale; w=b["w"]*scale; h=b["h"]*scale
        tx = (b["cx"]*scale); ty = y + 10*scale
        if not b["parts_only"]:
            draw.rounded_rectangle([x,y,x+w,y+h], radius=int(6*scale),
                                   outline=(0,0,0), width=2, fill=(253,253,253))
        fnt = font_part if b["parts_only"] else font_norm
        lh  = (b["line_height"]+1 if b["parts_only"] else b["line_height"])*scale
        for line in b["lines"]:
            # keep the leading '*' on partitions for quick identification
            bbox = draw.textbbox((0,0), line, font=fnt)
            tw = bbox[2]-bbox[0]
            draw.text((tx - tw/2, ty), line, fill=(0,0,0), font=fnt)
            ty += lh

    # save
    stem = os.path.join(OUTPUT_DIR, f"hasse_sl5_e{e}_adjusted")
    img.save(stem + ".png", "PNG")
    ImageOps.expand(img, border=10, fill="white").save(stem + ".pdf", "PDF", resolution=300.0)
    print("wrote:", stem + ".png", "and", stem + ".pdf")

if __name__ == "__main__":
    for e in (9,10,11):
        render_adjusted_sl5(e)


wrote: ./hasse_sl5_adjusted/hasse_sl5_e9_adjusted.png and ./hasse_sl5_adjusted/hasse_sl5_e9_adjusted.pdf
wrote: ./hasse_sl5_adjusted/hasse_sl5_e10_adjusted.png and ./hasse_sl5_adjusted/hasse_sl5_e10_adjusted.pdf
wrote: ./hasse_sl5_adjusted/hasse_sl5_e11_adjusted.png and ./hasse_sl5_adjusted/hasse_sl5_e11_adjusted.pdf
